# ViralCut AI — Google Colab Runner

### Quick Start:
1. Select GPU: **Runtime → Change runtime type → T4 GPU**
2. Click: **Runtime → Run all** (or run cells sequentially with ▶️)

The runner will automatically clone the repository, install dependencies, launch the server, pre-warm local Whisper AI, and generate a temporary Cloudflare Quick Tunnel URL (`trycloudflare.com`). No API keys or account setup required.

In [ ]:
%%capture
# Everything runs silently inside this setup cell

import subprocess, time, re, os, sys

# Repository URL (Single configuration point)
REPO_URL = "https://github.com/itxunknown39-web/ViralCut-AI.git"
REPO_NAME = "ViralCut-AI"

# 1) Repository clone / directory update
if not os.path.exists(REPO_NAME):
    subprocess.run(["git", "clone", REPO_URL, REPO_NAME])
os.chdir(REPO_NAME)

# 2) System dependencies & Python requirements
subprocess.run(["apt-get", "-qq", "update"])
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"])
subprocess.run(["pip", "install", "-q", "requests"])

# 3) Cloudflare Quick Tunnel binary (no token, no account)
if not os.path.exists("cloudflared"):
    subprocess.run([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "cloudflared",
    ])
    subprocess.run(["chmod", "+x", "cloudflared"])

# 4) Start existing FastAPI server
server_log = open("server.log", "w", encoding="utf-8")
server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT,
)

# 5) Health check loop (waits for /health to avoid Error 1033)
import requests
ready = False
for attempt in range(90):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(5)

# 6) Start Cloudflare Quick Tunnel once server is ready
public_url = None
if ready:
    tunnel_log = open("tunnel.log", "w", encoding="utf-8")
    tunnel_proc = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=tunnel_log, stderr=subprocess.STDOUT,
    )
    for attempt in range(30):
        time.sleep(2)
        if os.path.exists("tunnel.log"):
            with open("tunnel.log", encoding="utf-8") as f:
                text = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", text)
            if match:
                public_url = match.group(0)
                break

In [ ]:
# Display public URL result
from IPython.display import HTML, display

if public_url:
    print("🚀 ViralCut AI is ready! Open your app:\n")
    print(public_url)
    display(HTML(f'''
    <div style="font-family: sans-serif; background: #0a0b10; border: 1px solid #00C9FF; border-radius: 12px; padding: 20px; color: #fff; max-width: 600px; margin: 10px 0;">
        <h3 style="margin-top:0; color:#00C9FF;">ViralCut AI — by Kamran AI</h3>
        <p style="color:#9aa1b2; font-size:14px;">Your application is running live in Google Colab.</p>
        <a href="{public_url}" target="_blank" style="display: inline-block; background: #00C9FF; color: #031422; font-weight: bold; padding: 12px 24px; border-radius: 8px; text-decoration: none;">🚀 Open Dashboard</a>
    </div>
    '''))
elif not ready:
    print("❌ Server failed to start. Last 40 lines of server.log:\n")
    !tail -n 40 server.log
else:
    print("⚠️ Tunnel link pending. Last 40 lines of tunnel.log:\n")
    !tail -n 40 tunnel.log

---
💡 **Note**: The public URL is temporary and will change whenever the Colab session or tunnel restarts.